# Mamba State Analysis - Comprehensive Suite

This notebook consolidates all Mamba2 state analysis experiments:
- **Exp1**: Head heterogeneity & effective rank analysis
- **Exp2**: State similarity for semantic separability
- **Exp3**: State injection validation (enhanced with MSE)
- **Exp3-2**: State linearity tests (NEW)
- **Exp4**: Document position effect on QA (NEW)

Primary model: `state-spaces/mamba2-370m`

Forward-compatible design for Mamba3 when available.

## 0. Setup & Installation

In [ ]:
# Installation (run if needed)
import sys, subprocess, os

# 1. causal-conv1d
subprocess.run([sys.executable, '-m', 'pip', 'install',
              'git+https://github.com/Dao-AILab/causal-conv1d.git',
              '--no-build-isolation'], check=True)

# 2. mamba
os.chdir('/root')
subprocess.run(['rm', '-rf', 'mamba'], check=False)
subprocess.run(['git', 'clone', 'https://github.com/state-spaces/mamba.git'],
check=True)
os.chdir('/root/mamba')

env = os.environ.copy()
env.update({'CAUSAL_CONV1D_FORCE_BUILD': 'TRUE',
          'CAUSAL_CONV1D_SKIP_CUDA_BUILD': 'TRUE',
          'CAUSAL_CONV1D_FORCE_CXX11_ABI': 'TRUE'})

subprocess.run([sys.executable, '-m', 'pip', 'install',
'--no-build-isolation', '.'],
             env=env, check=True)
print("✅ 완료")

!pip install matplotlib seaborn scipy scikit-learn tqdm

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
from mamba_ssm.utils.generation import InferenceParams
from scipy.stats import ttest_rel, ttest_ind
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.09 GB


## 1. Model Configuration

In [3]:
# Model selection (enable/disable as needed)
MODEL_LIST = [
    "state-spaces/mamba2-370m",     # Primary test model
    # "state-spaces/mamba2-130m",   # Uncomment to enable
    # "state-spaces/mamba2-780m",
    # "state-spaces/mamba2-1.3b",
    # "state-spaces/mamba2-2.7b",
]

# Global parameters
T_RANGE = [32, 64, 128, 192, 256, 320, 384, 512, 768, 1024]
T_MAX = 1024
DOC_LEN = 256  # For exp2/exp3

# Head classification thresholds
SLOW_THRESHOLD = 0.99  # Type A
FAST_THRESHOLD = 0.50  # Type C

# Sampling
N_SAMPLES = 20
N_TOPICS = 5
N_DOCS_PER_TOPIC = 6
N_POSITION_DOCS = 10

# Results directory
RESULTS_DIR = "/Users/smaller225/code/state_memory/mamba3_analysis/results"
PLOTS_DIR = f"{RESULTS_DIR}/plots"

# Create directories if they don't exist
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f"Models to analyze: {len(MODEL_LIST)}")
print(f"Primary model: {MODEL_LIST[0]}")
print(f"Results will be saved to: {RESULTS_DIR}")

Models to analyze: 1
Primary model: state-spaces/mamba2-370m
Results will be saved to: /Users/smaller225/code/state_memory/mamba3_analysis/results


## 2. Core Utility Functions

In [4]:
def effective_rank(matrix):
    """
    Compute effective rank using Shannon entropy.
    
    Args:
        matrix: (headdim, d_state) numpy array or tensor
    
    Returns:
        float: effective rank
    """
    if isinstance(matrix, torch.Tensor):
        matrix = matrix.cpu().numpy()
    
    s = np.linalg.svd(matrix, compute_uv=False)
    s = s / (s.sum() + 1e-12)
    entropy = -np.sum(s * np.log(s + 1e-12))
    rank = np.exp(entropy)
    
    return rank

In [5]:
def get_ssm_states(model, input_ids):
    """
    InferenceParams를 통해 각 layer의 ssm_state 추출.
    
    Args:
        model: Mamba model
        input_ids: (batch, seq_len) tensor
    
    Returns:
        states: {layer_idx: Tensor(nheads, headdim, d_state)}
    """
    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)
    
    inference_params = InferenceParams(
        max_seqlen=input_ids.shape[1],
        max_batch_size=input_ids.shape[0]
    )
    
    with torch.no_grad():
        _ = model(input_ids, inference_params=inference_params)
    
    states = {}
    for layer_idx, (conv_state, ssm_state) in inference_params.key_value_memory_dict.items():
        # ssm_state: (batch, nheads, headdim, d_state) → squeeze batch
        states[layer_idx] = ssm_state.squeeze(0).cpu().float()
    
    return states

In [6]:
def get_A_disc(model):
    """
    Extract discrete-time decay parameters A_disc from all SSM layers.
    Mamba2 uses A_log, so A_disc = exp(A_log).
    
    Returns:
        np.array: (n_layers, n_heads) array of A_disc values
    """
    A_disc_all = []
    
    for layer in model.backbone.layers:
        if hasattr(layer, 'mixer') and hasattr(layer.mixer, 'A_log'):
            # A_log shape: (n_heads,)
            A_log = layer.mixer.A_log.detach()
            A_disc = torch.exp(A_log).cpu().numpy()
            A_disc_all.append(A_disc)
        elif hasattr(layer, 'mixer') and hasattr(layer.mixer, 'A'):
            # Fallback for Mamba1 or other variants
            A = layer.mixer.A.detach().cpu().numpy()
            if A.ndim == 2:
                A = A.mean(axis=1)
            A_disc_all.append(A)
    
    return np.array(A_disc_all)  # (n_layers, n_heads)

In [7]:
def classify_heads(A_disc, slow_threshold=0.99, fast_threshold=0.50):
    """
    Classify heads into Type A (slow), B (medium), C (fast).
    
    Args:
        A_disc: (n_layers, n_heads) array
    
    Returns:
        dict: {'A': mask, 'B': mask, 'C': mask}
    """
    type_A = A_disc >= slow_threshold
    type_C = A_disc <= fast_threshold
    type_B = ~(type_A | type_C)
    
    return {
        'A': type_A,
        'B': type_B,
        'C': type_C
    }

In [8]:
def extract_state(model, tokenizer, text, device='cuda'):
    """
    Extract state for a given text (for injection experiments).
    
    Returns:
        dict: {layer_idx: state_tensor (nheads, headdim, d_state)}
    """
    input_ids = tokenizer(text, return_tensors='pt')['input_ids'].to(device)
    return get_ssm_states(model, input_ids)

In [9]:
def inject_state_and_generate(model, tokenizer, query, state_dict, max_new_tokens=50, device='cuda'):
    """
    Generate text using injected state.
    
    Args:
        model: Mamba model
        tokenizer: tokenizer
        query: query string
        state_dict: {layer_idx: state_tensor (nheads, headdim, d_state)}
        max_new_tokens: generation length
    
    Returns:
        str: generated text
    """
    query_ids = tokenizer(query, return_tensors='pt')['input_ids'].to(device)
    batch_size = query_ids.shape[0]
    
    # Create inference params
    inference_params = InferenceParams(
        max_seqlen=query_ids.shape[1] + max_new_tokens,
        max_batch_size=batch_size
    )
    
    # Inject states as (conv_state=None, ssm_state)
    for layer_idx, ssm_state in state_dict.items():
        # ssm_state needs batch dimension: (1, nheads, headdim, d_state)
        ssm_state_batched = ssm_state.unsqueeze(0).to(device)
        # conv_state can be None
        inference_params.key_value_memory_dict[layer_idx] = (None, ssm_state_batched)
    
    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            query_ids,
            max_length=query_ids.shape[1] + max_new_tokens,
            inference_params=inference_params,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            return_dict_in_generate=False
        )
    
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [10]:
def compute_state_mse(state1, state2):
    """
    Compute MSE between two state dictionaries.
    
    Returns:
        dict: {layer_idx: mse_value}
        float: mean MSE across all layers
    """
    mse_per_layer = {}
    
    for layer_idx in state1.keys():
        if layer_idx in state2:
            s1 = state1[layer_idx].float()
            s2 = state2[layer_idx].float()
            mse = torch.mean((s1 - s2) ** 2).item()
            mse_per_layer[layer_idx] = mse
    
    mean_mse = np.mean(list(mse_per_layer.values())) if mse_per_layer else 0.0
    
    return mse_per_layer, mean_mse

In [11]:
def compute_state_cosine(state1, state2):
    """
    Compute cosine similarity between two state dictionaries.
    
    Returns:
        dict: {layer_idx: cosine_sim}
        float: mean cosine similarity
    """
    cos_per_layer = {}
    
    for layer_idx in state1.keys():
        if layer_idx in state2:
            s1 = state1[layer_idx].flatten().cpu().numpy()
            s2 = state2[layer_idx].flatten().cpu().numpy()
            cos = cosine_similarity(s1.reshape(1, -1), s2.reshape(1, -1))[0, 0]
            cos_per_layer[layer_idx] = cos
    
    mean_cos = np.mean(list(cos_per_layer.values())) if cos_per_layer else 0.0
    
    return cos_per_layer, mean_cos

In [12]:
def find_saturation_point(rank_trajectory, threshold_ratio=0.95):
    """
    Find T* where rank reaches 95% of maximum.
    
    Args:
        rank_trajectory: array of ranks at different T
        threshold_ratio: ratio of max rank to consider saturated
    
    Returns:
        int: T* (index in T_RANGE)
    """
    max_rank = np.max(rank_trajectory)
    threshold = threshold_ratio * max_rank
    
    for idx, rank in enumerate(rank_trajectory):
        if rank >= threshold:
            return idx
    
    return len(rank_trajectory) - 1

In [13]:
def check_keywords(text, keywords):
    """
    Check if any keyword is present in text (case-insensitive).
    
    Returns:
        bool: True if at least one keyword found
    """
    text_lower = text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

In [14]:
def clear_memory():
    """Clear CUDA cache and run garbage collection."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

print("✓ All utility functions defined successfully!")

✓ All utility functions defined successfully!


## 3. Data Preparation

In [15]:
# Sample texts for Exp1 (rank trajectory analysis)
SAMPLE_TEXTS = [
    "The transformer architecture revolutionized natural language processing.",
    "Machine learning models require large amounts of training data.",
    "Deep neural networks consist of multiple layers of interconnected neurons.",
    "Reinforcement learning agents learn by interacting with their environment.",
    "Computer vision systems can detect and classify objects in images.",
    "Natural language generation produces human-like text automatically.",
    "Gradient descent optimizes neural network parameters iteratively.",
    "Convolutional layers extract spatial features from image data.",
]

print(f"Prepared {len(SAMPLE_TEXTS)} sample texts")

Prepared 8 sample texts


In [16]:
# Topic documents for Exp2 (semantic separability)
TOPIC_DOCS = {
    "AI/ML": [
        "Deep learning models use neural networks with multiple layers to learn hierarchical representations.",
        "Convolutional neural networks excel at image recognition tasks through spatial feature extraction.",
        "Recurrent neural networks process sequential data by maintaining hidden states across time steps.",
        "Transfer learning leverages pre-trained models to improve performance on related tasks.",
        "Reinforcement learning optimizes agent behavior through reward signals from the environment.",
        "Attention mechanisms allow models to focus on relevant parts of input sequences dynamically.",
    ],
    "Space": [
        "The James Webb Space Telescope observes distant galaxies in infrared wavelengths.",
        "Mars rovers explore the red planet's surface searching for signs of ancient water.",
        "Exoplanets orbit stars beyond our solar system, potentially harboring alien life.",
        "Black holes warp spacetime so intensely that not even light can escape.",
        "The International Space Station orbits Earth every 90 minutes at 400 kilometers altitude.",
        "Gravitational waves from colliding neutron stars reveal the universe's violent events.",
    ],
    "History": [
        "The Roman Empire expanded across Europe, North Africa, and the Middle East.",
        "The Renaissance period saw a rebirth of art, science, and cultural achievement.",
        "The Industrial Revolution transformed society through mechanization and urbanization.",
        "Ancient Egyptian pyramids demonstrate advanced engineering and astronomical knowledge.",
        "The Silk Road facilitated trade and cultural exchange between East and West.",
        "The printing press revolutionized information dissemination in medieval Europe.",
    ],
    "Science": [
        "Quantum mechanics describes particle behavior at atomic and subatomic scales.",
        "DNA double helix structure encodes genetic information in all living organisms.",
        "Photosynthesis converts light energy into chemical energy in plant cells.",
        "The periodic table organizes elements by atomic number and chemical properties.",
        "Plate tectonics explains continental drift and earthquake formation mechanisms.",
        "Evolution through natural selection drives species adaptation over generations.",
    ],
    "Geography": [
        "The Amazon rainforest produces 20% of Earth's oxygen and hosts incredible biodiversity.",
        "Mount Everest rises 8,849 meters above sea level in the Himalayan mountain range.",
        "The Sahara Desert spans 9 million square kilometers across North Africa.",
        "The Nile River flows 6,650 kilometers from Uganda to the Mediterranean Sea.",
        "Antarctica contains 90% of Earth's ice and experiences extreme polar conditions.",
        "The Great Barrier Reef stretches 2,300 kilometers along Australia's coast.",
    ],
}

print(f"Prepared {len(TOPIC_DOCS)} topics × {len(TOPIC_DOCS['AI/ML'])} docs = {sum(len(v) for v in TOPIC_DOCS.values())} total docs")

Prepared 5 topics × 6 docs = 30 total docs


In [17]:
# QA pairs for Exp3 (state injection)
QA_PAIRS = [
    {
        "document": "Apollo 11 was the spaceflight that first landed humans on the Moon. Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed the Apollo Lunar Module Eagle on July 20, 1969. Armstrong became the first person to step onto the lunar surface six hours later.",
        "question": "Who was the first person to walk on the Moon?",
        "keywords": ["Neil Armstrong", "Armstrong", "Neil"],
    },
    {
        "document": "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to create oxygen and energy in the form of sugar. Chlorophyll, the green pigment in leaves, absorbs light energy which drives the chemical reactions.",
        "question": "What pigment in leaves absorbs light energy?",
        "keywords": ["chlorophyll", "Chlorophyll"],
    },
    {
        "document": "The Eiffel Tower was built by Gustave Eiffel for the 1889 World's Fair in Paris. Standing 330 meters tall, it was the world's tallest structure until 1930. Made of wrought iron, it weighs approximately 10,000 tons.",
        "question": "Who built the Eiffel Tower?",
        "keywords": ["Gustave Eiffel", "Eiffel", "Gustave"],
    },
    {
        "document": "Python is a high-level programming language created by Guido van Rossum in 1991. Known for its simple syntax and readability, Python has become one of the most popular languages for web development, data science, and artificial intelligence.",
        "question": "Who created the Python programming language?",
        "keywords": ["Guido van Rossum", "van Rossum", "Guido"],
    },
    {
        "document": "Mount Everest, located in the Himalayas on the border between Nepal and Tibet, is Earth's highest mountain at 8,849 meters above sea level. Sir Edmund Hillary and Tenzing Norgay were the first confirmed climbers to reach the summit in 1953.",
        "question": "Who were the first people to reach the summit of Mount Everest?",
        "keywords": ["Hillary", "Norgay", "Edmund Hillary", "Tenzing Norgay"],
    },
]

print(f"Prepared {len(QA_PAIRS)} QA pairs")

Prepared 5 QA pairs


## 6. Experiment 3: State Injection Validation

This experiment validates the RAG mechanism via state injection:

**Test setups**:
- A (Oracle): [doc + query] → forward → generate
- B (Inject): [doc → state] → inject + query → generate
- C (No context): [query only] → generate
- D (Wrong context): [irrelevant state + query] → generate

**Enhanced metrics**:
- Keyword matching (hit/miss)
- **NEW**: MSE between Oracle and Injected final states
- Full answer display

In [18]:
print("="*60)
print("Experiment 3: State Injection Validation")
print("="*60)

# Use primary model
model_name = MODEL_LIST[0]
print(f"\\nLoading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
tokenizer.pad_token = tokenizer.eos_token

# Load model
model = MambaLMHeadModel.from_pretrained(model_name, device=device, dtype=torch.float32)
model.eval()

print("Model loaded successfully!")

Experiment 3: State Injection Validation
\nLoading model: state-spaces/mamba2-370m
Model loaded successfully!


In [19]:
# Storage for results
exp3_results = {
    'A_oracle': [],
    'B_inject': [],
    'C_no_context': [],
    'D_wrong': [],
    'oracle_inject_mse': [],
}

# Wrong context (use first QA doc for all as irrelevant)
wrong_doc = QA_PAIRS[0]['document']
wrong_state = extract_state(model, tokenizer, wrong_doc, device=device)

print("\\nRunning state injection tests...")
print(f"Number of QA pairs: {len(QA_PAIRS)}\\n")

for qa_idx, qa in enumerate(tqdm(QA_PAIRS, desc="QA pairs")):
    document = qa['document']
    question = qa['question']
    keywords = qa['keywords']

    print(f"\\n--- QA {qa_idx + 1} ---")
    print(f"Question: {question}")

    # Setup A: Oracle (doc + query forward)
    combined_text = document + " " + question
    input_ids = tokenizer(combined_text, return_tensors='pt')['input_ids'].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False
        )

    answer_A = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Extract only the generated part (after question)
    answer_A_only = answer_A[len(combined_text):].strip()
    hit_A = check_keywords(answer_A_only, keywords)

    # Also extract final state for MSE comparison
    states_A_final = get_ssm_states(model, output_ids)

    # Setup B: Injection (doc → state, then inject + query)
    doc_state = extract_state(model, tokenizer, document, device=device)
    answer_B = inject_state_and_generate(model, tokenizer, question, doc_state, max_new_tokens=50, device=device)
    answer_B_only = answer_B[len(question):].strip()
    hit_B = check_keywords(answer_B_only, keywords)

    # Extract final state after injection for MSE
    query_ids = tokenizer(question, return_tensors='pt')['input_ids'].to(device)
    inference_params = InferenceParams(max_seqlen=query_ids.shape[1] + 50, max_batch_size=1)

    # Inject state
    for layer_idx, state in doc_state.items():
        key = (layer_idx, 0)
        inference_params.key_value_memory_dict[key] = state

    # Generate
    with torch.no_grad():
        output_ids_B = model.generate(
            query_ids,
            max_new_tokens=50,
            inference_params=inference_params,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False
        )

    states_B_final = get_ssm_states(model, output_ids_B)

    # Compute MSE between A and B final states
    mse_per_layer, mean_mse = compute_state_mse(states_A_final, states_B_final)

    # Setup C: No context
    query_only_ids = tokenizer(question, return_tensors='pt')['input_ids'].to(device)
    with torch.no_grad():
        output_ids_C = model.generate(
            query_only_ids,
            max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False
        )

    answer_C = tokenizer.decode(output_ids_C[0], skip_special_tokens=True)
    answer_C_only = answer_C[len(question):].strip()
    hit_C = check_keywords(answer_C_only, keywords)

    # Setup D: Wrong context
    answer_D = inject_state_and_generate(model, tokenizer, question, wrong_state, max_new_tokens=50, device=device)
    answer_D_only = answer_D[len(question):].strip()
    hit_D = check_keywords(answer_D_only, keywords)

    # Store results
    exp3_results['A_oracle'].append({
        'question': question,
        'answer': answer_A_only,
        'hit': hit_A,
        'keywords': keywords
    })

    exp3_results['B_inject'].append({
        'question': question,
        'answer': answer_B_only,
        'hit': hit_B,
        'keywords': keywords
    })

    exp3_results['C_no_context'].append({
        'question': question,
        'answer': answer_C_only,
        'hit': hit_C,
        'keywords': keywords
    })

    exp3_results['D_wrong'].append({
        'question': question,
        'answer': answer_D_only,
        'hit': hit_D,
        'keywords': keywords
    })

    exp3_results['oracle_inject_mse'].append({
        'mse_per_layer': mse_per_layer,
        'mean_mse': mean_mse
    })

    print(f"  A (Oracle): {'HIT' if hit_A else 'MISS'}")
    print(f"  B (Inject): {'HIT' if hit_B else 'MISS'} | MSE: {mean_mse:.6f}")
    print(f"  C (No ctx): {'HIT' if hit_C else 'MISS'}")
    print(f"  D (Wrong):  {'HIT' if hit_D else 'MISS'}")

# Compute aggregate hit rates
hit_rate_A = sum(r['hit'] for r in exp3_results['A_oracle']) / len(QA_PAIRS)
hit_rate_B = sum(r['hit'] for r in exp3_results['B_inject']) / len(QA_PAIRS)
hit_rate_C = sum(r['hit'] for r in exp3_results['C_no_context']) / len(QA_PAIRS)
hit_rate_D = sum(r['hit'] for r in exp3_results['D_wrong']) / len(QA_PAIRS)

print("\\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(f"A (Oracle):      {hit_rate_A:.2%} hit rate")
print(f"B (Inject):      {hit_rate_B:.2%} hit rate")
print(f"C (No context):  {hit_rate_C:.2%} hit rate")
print(f"D (Wrong):       {hit_rate_D:.2%} hit rate")

mean_mse_all = np.mean([r['mean_mse'] for r in exp3_results['oracle_inject_mse']])
print(f"\\nOracle-Inject Mean MSE: {mean_mse_all:.6f}")

# Save results
with open(f"{RESULTS_DIR}/exp3_results.json", 'w') as f:
    # Convert numpy types for JSON
    save_dict = {}
    for key in ['A_oracle', 'B_inject', 'C_no_context', 'D_wrong']:
        save_dict[key] = exp3_results[key]

    save_dict['oracle_inject_mse'] = [
        {
            'mse_per_layer': {int(k): float(v) for k, v in r['mse_per_layer'].items()},
            'mean_mse': float(r['mean_mse'])
        }
        for r in exp3_results['oracle_inject_mse']
    ]

    save_dict['summary'] = {
        'hit_rate_A': float(hit_rate_A),
        'hit_rate_B': float(hit_rate_B),
        'hit_rate_C': float(hit_rate_C),
        'hit_rate_D': float(hit_rate_D),
        'mean_mse': float(mean_mse_all)
    }

    json.dump(save_dict, f, indent=2)

print(f"\\nResults saved to {RESULTS_DIR}/exp3_results.json")

\nRunning state injection tests...
Number of QA pairs: 5\n


QA pairs:   0%|          | 0/5 [00:00<?, ?it/s]

\n--- QA 1 ---
Question: Who was the first person to walk on the Moon?


TypeError: GenerationMixin.generate() missing 1 required positional argument: 'max_length'

In [ ]:
print("\\nGenerating visualizations...")

# Hit rate bar chart
fig, ax = plt.subplots(figsize=(10, 6))

setups = ['A\\n(Oracle)', 'B\\n(Inject)', 'C\\n(No ctx)', 'D\\n(Wrong)']
hit_rates = [hit_rate_A, hit_rate_B, hit_rate_C, hit_rate_D]
colors = ['steelblue', 'green', 'orange', 'salmon']

bars = ax.bar(setups, hit_rates, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, rate in zip(bars, hit_rates):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.02, f'{rate:.1%}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Hit Rate', fontsize=12)
ax.set_title(f'State Injection Validation\\n{model_name}', fontsize=14)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_path = f"{PLOTS_DIR}/exp3_hit_rates.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved to {save_path}")
plt.show()

In [ ]:
# MSE heatmap (Layer × QA pair)
n_layers = len(model.backbone.layers)
mse_matrix = np.zeros((n_layers, len(QA_PAIRS)))

for qa_idx, mse_result in enumerate(exp3_results['oracle_inject_mse']):
    for layer_idx, mse_val in mse_result['mse_per_layer'].items():
        mse_matrix[layer_idx, qa_idx] = mse_val

fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(mse_matrix, cmap='viridis', aspect='auto')
ax.set_xlabel('QA Pair Index', fontsize=12)
ax.set_ylabel('Layer Index', fontsize=12)
ax.set_title(f'Oracle vs Inject State MSE\\n{model_name}', fontsize=14)
ax.set_xticks(range(len(QA_PAIRS)))
ax.set_yticks(range(n_layers))
plt.colorbar(im, ax=ax, label='MSE')

plt.tight_layout()
save_path = f"{PLOTS_DIR}/exp3_mse_heatmap.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved to {save_path}")
plt.show()

In [ ]:
print("\\n" + "="*60)
print("FULL ANSWER COMPARISON")
print("="*60)

for qa_idx in range(len(QA_PAIRS)):
    print(f"\\n### QA Pair {qa_idx + 1}")
    print(f"**Question**: {exp3_results['A_oracle'][qa_idx]['question']}")
    print(f"**Keywords**: {exp3_results['A_oracle'][qa_idx]['keywords']}")
    print()

    setups = ['A_oracle', 'B_inject', 'C_no_context', 'D_wrong']
    labels = ['Oracle', 'Inject', 'No Context', 'Wrong']

    for setup, label in zip(setups, labels):
        result = exp3_results[setup][qa_idx]
        hit_mark = '✓' if result['hit'] else '✗'
        print(f"**{label}**: {result['answer'][:100]}... [{hit_mark}]")

    if qa_idx < len(exp3_results['oracle_inject_mse']):
        mse = exp3_results['oracle_inject_mse'][qa_idx]['mean_mse']
        print(f"**Oracle-Inject MSE**: {mse:.6f}")

    print()

print("\\nExperiment 3 complete!")

## 7. Experiment 3-2: State Linearity Tests

This experiment tests whether SSM state operations are linear/additive:

**Test cases**:
1. **Addition**: `state(doc1 + doc2) ≟ state(doc1) + state(doc2) - state(init)`
2. **Subtraction**: `state(doc1) ≟ state(doc1+doc2) - state(doc2) + state(init)`
3. **Weighted merge**: `state_merged = α·state(doc1) + (1-α)·state(doc2)`
4. **Generation quality**: Check if merged states produce coherent output

**Metrics**: MSE, cosine similarity, keyword matching

In [ ]:
print("="*60)
print("Experiment 3-2: State Linearity Tests")
print("="*60)

# Create diverse documents for linearity testing
LINEARITY_DOCS = [
    # Doc 0: Space
    "Apollo 11 was the spaceflight that landed the first humans on the Moon in 1969. Neil Armstrong and Buzz Aldrin became the first people to walk on the lunar surface.",

    # Doc 1: Programming
    "Python is a high-level programming language created by Guido van Rossum. It emphasizes code readability with its notable use of significant indentation.",

    # Doc 2: Ecology
    "The Amazon rainforest produces approximately 20% of Earth's oxygen. It hosts an estimated 390 billion trees and countless species of plants and animals.",

    # Doc 3: Music
    "Wolfgang Amadeus Mozart was a prolific composer of the Classical era. He composed over 600 works including symphonies, operas, and concertos.",

    # Doc 4: Sports
    "Tennis is played on a rectangular court with a net stretched across the center. Players use rackets to hit a ball over the net into the opponent's court.",

    # Doc 5: Chemistry
    "Photosynthesis is the process by which plants convert light energy into chemical energy. Chlorophyll absorbs sunlight to produce glucose and oxygen.",

    # Doc 6: AI
    "Transformer models use self-attention mechanisms to process sequential data. They have revolutionized natural language processing since their introduction in 2017.",

    # Doc 7: Astronomy
    "Mars is the fourth planet from the Sun and has been explored by numerous rovers. Scientists study it for signs of past water and potential for life.",
]

print(f"Prepared {len(LINEARITY_DOCS)} diverse documents for testing")

In [ ]:
# Use primary model
model_name = MODEL_LIST[0]
print(f"\\nLoading model: {model_name}")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=torch.float32,
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
n_layers = len(model.backbone.layers)

print("Model loaded successfully!")

In [ ]:
def state_add(state1, state2, weight2=1.0, state_init=None):
    """
    Add two states: state1 + weight2 * state2 - state_init

    Args:
        state1, state2: state dictionaries
        weight2: weight for state2 (default 1.0)
        state_init: initial state to subtract (for proper addition)

    Returns:
        dict: merged state
        """

    merged = {}
    for layer_idx in state1.keys():
        if layer_idx in state2:
            merged_layer = state1[layer_idx] + weight2 * state2[layer_idx]
            if state_init is not None and layer_idx in state_init:
                merged_layer = merged_layer - state_init[layer_idx]
            merged[layer_idx] = merged_layer
    return merged

def state_subtract(state1, state2, state_init=None):
    """
    Subtract states: state1 - state2 + state_init
    """
    diff = {}
    for layer_idx in state1.keys():
        if layer_idx in state2:
            diff_layer = state1[layer_idx] - state2[layer_idx]
            if state_init is not None and layer_idx in state_init:
                diff_layer = diff_layer + state_init[layer_idx]
            diff[layer_idx] = diff_layer
    return diff

def weighted_merge(state1, state2, alpha):
    """
    Weighted merge: alpha * state1 + (1-alpha) * state2
    """
    merged = {}
    for layer_idx in state1.keys():
        if layer_idx in state2:
            merged[layer_idx] = alpha * state1[layer_idx] + (1 - alpha) * state2[layer_idx]
    return merged

print("State operation functions defined.")

### 7.2 Test Case 1: Addition

In [ ]:
print("\\n" + "="*50)
print("Test Case 1: Addition")
print("="*50)
print("Hypothesis: state(doc1 + doc2) ≈ state(doc1) + state(doc2) - state(init)")
print()

# Extract initial state (empty input)
state_init = extract_state(model, tokenizer, "", device=device)

# Test 5 document pairs
addition_pairs = [(0,1), (2,3), (4,5), (6,7), (0,7)]
addition_results = []

for pair_idx, (doc_i, doc_j) in enumerate(addition_pairs):
    print(f"\\nPair {pair_idx + 1}: Doc {doc_i} + Doc {doc_j}")

    doc1 = LINEARITY_DOCS[doc_i]
    doc2 = LINEARITY_DOCS[doc_j]
    combined = doc1 + " " + doc2

    # Extract states
    state1 = extract_state(model, tokenizer, doc1, device=device)
    state2 = extract_state(model, tokenizer, doc2, device=device)
    state_combined_true = extract_state(model, tokenizer, combined, device=device)

    # Linear prediction
    state_combined_pred = state_add(state1, state2, 1.0, state_init)

    # Compute metrics
    mse_per_layer, mean_mse = compute_state_mse(state_combined_true, state_combined_pred)
    cos_per_layer, mean_cos = compute_state_cosine(state_combined_true, state_combined_pred)

    print(f"  MSE: {mean_mse:.6f}")
    print(f"  Cosine: {mean_cos:.4f}")

    # Generation test
    query = "Summarize the context: "

    # True combined state generation
    gen_true = inject_state_and_generate(model, tokenizer, query, state_combined_true, max_new_tokens=40, device=device)
    gen_true_only = gen_true[len(query):].strip()

    # Predicted combined state generation
    gen_pred = inject_state_and_generate(model, tokenizer, query, state_combined_pred, max_new_tokens=40, device=device)
    gen_pred_only = gen_pred[len(query):].strip()

    print(f"  True gen: {gen_true_only[:60]}...")
    print(f"  Pred gen: {gen_pred_only[:60]}...")

    # Check if generations mention both topics
    # Extract key topic words from each doc
    doc1_keywords = set(doc1.lower().split()[:5])  # Simple heuristic
    doc2_keywords = set(doc2.lower().split()[:5])

    gen_true_words = set(gen_true_only.lower().split())
    gen_pred_words = set(gen_pred_only.lower().split())

    match_true = len(doc1_keywords & gen_true_words) + len(doc2_keywords & gen_true_words)
    match_pred = len(doc1_keywords & gen_pred_words) + len(doc2_keywords & gen_pred_words)

    print(f"  Keyword matches - True: {match_true}, Pred: {match_pred}")

    addition_results.append({
        'pair': (doc_i, doc_j),
        'mse': mean_mse,
        'cosine': mean_cos,
        'gen_true': gen_true_only,
        'gen_pred': gen_pred_only,
        'match_true': match_true,
        'match_pred': match_pred,
    })

print("\\nAddition test complete!")

### 7.3 Test Case 2: Subtraction

In [ ]:
print("\\n" + "="*50)
print("Test Case 2: Subtraction")
print("="*50)
print("Hypothesis: state(doc1) ≈ state(doc1+doc2) - state(doc2) + state(init)")
print()

# Test 3 document pairs
subtraction_pairs = [(0,1), (2,3), (4,5)]
subtraction_results = []

for pair_idx, (doc_i, doc_j) in enumerate(subtraction_pairs):
    print(f"\\nPair {pair_idx + 1}: Extract Doc {doc_i} from (Doc {doc_i} + Doc {doc_j})")

    doc1 = LINEARITY_DOCS[doc_i]
    doc2 = LINEARITY_DOCS[doc_j]
    combined = doc1 + " " + doc2

    # Extract states
    state1_true = extract_state(model, tokenizer, doc1, device=device)
    state2 = extract_state(model, tokenizer, doc2, device=device)
    state_combined = extract_state(model, tokenizer, combined, device=device)

    # Subtraction: try to recover doc1 from combined
    state1_pred = state_subtract(state_combined, state2, state_init)

    # Compute metrics
    mse_per_layer, mean_mse = compute_state_mse(state1_true, state1_pred)
    cos_per_layer, mean_cos = compute_state_cosine(state1_true, state1_pred)

    print(f"  MSE: {mean_mse:.6f}")
    print(f"  Cosine: {mean_cos:.4f}")

    # Generation test: Does reconstructed state produce doc1 content?
    query = "What was in the context?"

    gen_true = inject_state_and_generate(model, tokenizer, query, state1_true, max_new_tokens=40, device=device)
    gen_true_only = gen_true[len(query):].strip()

    gen_pred = inject_state_and_generate(model, tokenizer, query, state1_pred, max_new_tokens=40, device=device)
    gen_pred_only = gen_pred[len(query):].strip()

    print(f"  True gen: {gen_true_only[:60]}...")
    print(f"  Pred gen: {gen_pred_only[:60]}...")

    # Check if doc1 content is preserved
    doc1_keywords = set(doc1.lower().split()[:8])
    match_true = len(doc1_keywords & set(gen_true_only.lower().split()))
    match_pred = len(doc1_keywords & set(gen_pred_only.lower().split()))

    print(f"  Doc1 keyword matches - True: {match_true}, Pred: {match_pred}")

    subtraction_results.append({
        'pair': (doc_i, doc_j),
        'mse': mean_mse,
        'cosine': mean_cos,
        'gen_true': gen_true_only,
        'gen_pred': gen_pred_only,
        'match_true': match_true,
        'match_pred': match_pred,
    })

print("\\nSubtraction test complete!")

### 7.4 Test Case 3: Weighted Merge

In [ ]:
print("\\n" + "="*50)
print("Test Case 3: Weighted Merge")
print("="*50)
print("Hypothesis: α controls information balance in merged state")
print()

# Test 3 pairs × 3 alpha values
weighted_pairs = [(0,1), (2,3), (4,5)]
alphas = [0.3, 0.5, 0.7]
weighted_results = []

for pair_idx, (doc_i, doc_j) in enumerate(weighted_pairs):
    print(f"\\n--- Pair {pair_idx + 1}: Doc {doc_i} + Doc {doc_j} ---")

    doc1 = LINEARITY_DOCS[doc_i]
    doc2 = LINEARITY_DOCS[doc_j]

    state1 = extract_state(model, tokenizer, doc1, device=device)
    state2 = extract_state(model, tokenizer, doc2, device=device)

    # Create questions about each document
    query1 = f"What is the main topic of the first document?"
    query2 = f"What is the main topic of the second document?"

    for alpha in alphas:
        print(f"\\n  Alpha = {alpha}")

        # Weighted merge
        state_merged = weighted_merge(state1, state2, alpha)

        # Query both documents
        ans1 = inject_state_and_generate(model, tokenizer, query1, state_merged, max_new_tokens=30, device=device)
        ans1_only = ans1[len(query1):].strip()

        ans2 = inject_state_and_generate(model, tokenizer, query2, state_merged, max_new_tokens=30, device=device)
        ans2_only = ans2[len(query2):].strip()

        print(f"    Doc1 answer: {ans1_only[:50]}...")
        print(f"    Doc2 answer: {ans2_only[:50]}...")

        # Score: Does answer quality match alpha weighting?
        doc1_keywords = set(doc1.lower().split()[:8])
        doc2_keywords = set(doc2.lower().split()[:8])

        score1 = len(doc1_keywords & set(ans1_only.lower().split()))
        score2 = len(doc2_keywords & set(ans2_only.lower().split()))

        print(f"    Scores - Doc1: {score1}, Doc2: {score2}")

        # Expected: higher alpha → better score1
        weighted_results.append({
            'pair': (doc_i, doc_j),
            'alpha': alpha,
            'ans1': ans1_only,
            'ans2': ans2_only,
            'score1': score1,
            'score2': score2,
        })

print("\\nWeighted merge test complete!")

### 7.6 Visualization

In [ ]:
print("\\nGenerating visualizations...")

# Heatmap 1: Addition test MSE (pairs × layers)
# Extract MSE per layer for each pair
addition_mse_matrix = np.zeros((len(addition_pairs), n_layers))

for pair_idx, (doc_i, doc_j) in enumerate(addition_pairs):
    doc1 = LINEARITY_DOCS[doc_i]
    doc2 = LINEARITY_DOCS[doc_j]
    combined = doc1 + " " + doc2

    state1 = extract_state(model, tokenizer, doc1, device=device)
    state2 = extract_state(model, tokenizer, doc2, device=device)
    state_combined_true = extract_state(model, tokenizer, combined, device=device)
    state_combined_pred = state_add(state1, state2, 1.0, state_init)

    mse_per_layer, _ = compute_state_mse(state_combined_true, state_combined_pred)

    for layer_idx, mse_val in mse_per_layer.items():
        addition_mse_matrix[pair_idx, layer_idx] = mse_val

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(addition_mse_matrix, cmap='viridis', aspect='auto')
ax.set_xlabel('Layer Index', fontsize=12)
ax.set_ylabel('Document Pair', fontsize=12)
ax.set_title(f'Addition Test: MSE per Layer\\n{model_name}', fontsize=14)
ax.set_yticks(range(len(addition_pairs)))
ax.set_yticklabels([f'Pair {i+1}: ({p[0]},{p[1]})' for i, p in enumerate(addition_pairs)])
ax.set_xticks(range(n_layers))
plt.colorbar(im, ax=ax, label='MSE')

plt.tight_layout()
save_path = f"{PLOTS_DIR}/exp3_2_addition_mse.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved to {save_path}")
plt.show()

In [ ]:
# Bar chart: Mean MSE by operation type
fig, ax = plt.subplots(figsize=(10, 6))

operations = ['Addition', 'Subtraction']
mean_mses = [
    np.mean([r['mse'] for r in addition_results]),
    np.mean([r['mse'] for r in subtraction_results])
]

bars = ax.bar(operations, mean_mses, color=['steelblue', 'orange'], alpha=0.8, edgecolor='black')

for bar, mse in zip(bars, mean_mses):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.001, f'{mse:.4f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Mean MSE', fontsize=12)
ax.set_title(f'State Linearity: MSE by Operation\\n{model_name}', fontsize=14)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_path = f"{PLOTS_DIR}/exp3_2_mse_by_operation.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved to {save_path}")
plt.show()

In [ ]:
# Line plot: Alpha vs Answer Quality
fig, ax = plt.subplots(figsize=(10, 6))

# Aggregate scores by alpha
alpha_values = sorted(set(r['alpha'] for r in weighted_results))
score1_by_alpha = {a: [] for a in alpha_values}
score2_by_alpha = {a: [] for a in alpha_values}

for r in weighted_results:
    score1_by_alpha[r['alpha']].append(r['score1'])
    score2_by_alpha[r['alpha']].append(r['score2'])

mean_score1 = [np.mean(score1_by_alpha[a]) for a in alpha_values]
mean_score2 = [np.mean(score2_by_alpha[a]) for a in alpha_values]

ax.plot(alpha_values, mean_score1, 'o-', linewidth=2, markersize=10, label='Doc1 Quality', color='steelblue')
ax.plot(alpha_values, mean_score2, 's-', linewidth=2, markersize=10, label='Doc2 Quality', color='orange')

ax.set_xlabel('Alpha (Weight for Doc1)', fontsize=12)
ax.set_ylabel('Answer Quality (Keyword Matches)', fontsize=12)
ax.set_title(f'Weighted Merge: Alpha vs Quality\\n{model_name}', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xticks(alpha_values)

plt.tight_layout()
save_path = f"{PLOTS_DIR}/exp3_2_weighted_alpha.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved to {save_path}")
plt.show()

In [ ]:
exp3_2_results = {
    'addition': addition_results,
    'subtraction': subtraction_results,
    'weighted': weighted_results,
}

with open(f"{RESULTS_DIR}/exp3_2_linearity.json", 'w') as f:
    # Convert numpy types
    save_dict = {}
    for key, results in exp3_2_results.items():
        save_dict[key] = []
        for r in results:
            r_copy = r.copy()
            for k, v in r_copy.items():
                if isinstance(v, (np.int64, np.int32)):
                    r_copy[k] = int(v)
                elif isinstance(v, (np.float64, np.float32)):
                    r_copy[k] = float(v)
            save_dict[key].append(r_copy)

    json.dump(save_dict, f, indent=2)

print(f"\\nResults saved to {RESULTS_DIR}/exp3_2_linearity.json")
print("\\nExperiment 3-2 complete!")